# GNN Cloud Notebook: Architecture Comparison

This notebook is configured for Google Colab using GitHub as the code source. It clones the repo into `/content`, keeps Google Drive optional, and runs an architecture comparison on the same best confirmed training pipeline.

Compared models:
- `GCN-2 Control`: current best prior architecture baseline
- `GIN`
- `GraphSAGE`
- `GCN-3`

This comparison keeps the following fixed across models:
- Adam with the baseline two-phase schedule
- weighted edges where the architecture supports them
- richer node features derived at load time
- graph-level summary features concatenated after graph pooling
- standardized regression targets during training

No new lattice sets are required.
        


In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    %pip -q install torch-geometric
else:
    print('Colab dependency install cell skipped.')
        


In [ ]:
REPO_URL = 'https://github.com/aadams2006/NSF-REU-Summer-26.git'
REPO_DIR = '/content/NSF-REU-Summer-26'

if IN_COLAB:
    import os
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print(f'Repo already exists at {REPO_DIR}')
else:
    print('Git clone cell skipped outside Colab.')
        


In [ ]:
from datetime import datetime
from getpass import getpass
from pathlib import Path
import os
import sys

USE_DRIVE_FOR_DATA = False
SAVE_OUTPUTS_TO_DRIVE = True
PUSH_RESULTS_TO_GITHUB = False
PUSH_MODEL_TO_GITHUB = False
DRIVE_DATA_ROOT = '/content/drive/MyDrive/lattice_data'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/GCN_Cloud_Outputs_Architecture_Comparison'
GIT_RESULTS_SUBDIR = 'active_projects/voronoi_lattice_pipeline/gnn_prototype/GCN_Cloud_Outputs_Architecture_Comparison'
GIT_BRANCH = 'main'
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '').strip()
GIT_COMMIT_USERNAME = os.environ.get('GIT_COMMIT_USERNAME', '').strip()
GIT_COMMIT_EMAIL = os.environ.get('GIT_COMMIT_EMAIL', '').strip()
RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

if IN_COLAB:
    repo_root = Path(REPO_DIR).resolve()
else:
    repo_root = Path.cwd().resolve()

pipeline_root = repo_root / 'active_projects' / 'voronoi_lattice_pipeline'
module_dir = pipeline_root / 'gnn_prototype'
optimization_dir = module_dir / 'GCN_Optimization'
if not (module_dir / 'colab_gnn_stiffness_prototype.py').is_file():
    raise FileNotFoundError(f'Module not found at {module_dir}')
if not (optimization_dir / 'architecture_comparison_runner.py').is_file():
    raise FileNotFoundError(f'Runner not found at {optimization_dir}')

# Keep the notebook process rooted at the pipeline directory so any fallback
# path discovery inside shared helpers resolves correctly in Colab.
os.chdir(pipeline_root)

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))
if str(optimization_dir) not in sys.path:
    sys.path.insert(0, str(optimization_dir))

if IN_COLAB and (USE_DRIVE_FOR_DATA or SAVE_OUTPUTS_TO_DRIVE):
    from google.colab import drive
    drive.mount('/content/drive')

if USE_DRIVE_FOR_DATA:
    if not IN_COLAB:
        raise RuntimeError('USE_DRIVE_FOR_DATA is only supported in Colab.')
    drive_data_root = Path(DRIVE_DATA_ROOT)
    train_root = drive_data_root / 'Randomness_Sweep'
    predict_root = drive_data_root / 'Lattice_Guess_Prediction_Input_Data'
else:
    train_root = pipeline_root / 'source_archives' / 'lattice_data' / 'Randomness_Sweep'
    predict_root = pipeline_root / 'datasets' / 'Lattice_Guess_Prediction_Input_Data'

if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
    output_root = Path(DRIVE_OUTPUT_ROOT)
else:
    output_root = Path('/content/gnn_outputs_architecture_comparison') if IN_COLAB else pipeline_root / 'gnn_prototype' / 'outputs_architecture_comparison'

git_output_root = repo_root / GIT_RESULTS_SUBDIR
output_dir = output_root / f'run_{RUN_STAMP}'
per_model_output_root = output_dir / 'per_model'
output_dir.mkdir(parents=True, exist_ok=True)
(output_root / 'latest_run.txt').write_text(str(output_dir), encoding='utf-8')

if PUSH_RESULTS_TO_GITHUB:
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = getpass('Enter GitHub token: ').strip()
    if not GIT_COMMIT_USERNAME:
        GIT_COMMIT_USERNAME = input('Enter Git commit username or display name: ').strip()
    if not GIT_COMMIT_EMAIL:
        GIT_COMMIT_EMAIL = input('Enter Git commit email (GitHub noreply or verified email): ').strip()

print(f'Repo root: {repo_root}')
print(f'Pipeline root: {pipeline_root}')
print(f'Train data: {train_root}')
print(f'Prediction data: {predict_root}')
print(f'Output root: {output_root}')
print(f'Current run dir: {output_dir}')
print(f'Working directory: {Path.cwd()}')
print(f'Git output root: {git_output_root}')
print(f'Push results to GitHub: {PUSH_RESULTS_TO_GITHUB}')
print(f'GitHub token loaded: {bool(GITHUB_TOKEN)}')
print(f'Git commit username loaded: {bool(GIT_COMMIT_USERNAME)}')
print(f'Git commit email loaded: {bool(GIT_COMMIT_EMAIL)}')
        


In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
import shutil
import subprocess
from IPython.display import display

from architecture_comparison_runner import ArchitectureConfig, run_architecture_experiment
        


In [ ]:
COMPARISON_SEED = 42
HIDDEN_DIM = 24
ARCHITECTURES = [
    ('gcn2_control', 'GCN-2 Control (Best Prior)'),
    ('gin', 'GIN'),
    ('graphsage', 'GraphSAGE'),
    ('gcn3', 'GCN-3'),
]

print(f'Comparison seed: {COMPARISON_SEED}')
print(f'Hidden dim: {HIDDEN_DIM}')
for key, label in ARCHITECTURES:
    print(f' - {label}: {key}')
        


In [ ]:
experiment_results = {}
summary_frames = []

for architecture_name, architecture_label in ARCHITECTURES:
    config = ArchitectureConfig(
        architecture_name=architecture_name,
        architecture_label=architecture_label,
        hidden_dim=HIDDEN_DIM,
        seed=COMPARISON_SEED,
        output_group='architecture_comparison',
    )
    result = run_architecture_experiment(
        config,
        train_root=train_root,
        predict_root=predict_root,
        output_root=per_model_output_root,
    )
    experiment_results[architecture_name] = result

    summary_frame = result['summary_frame'].copy()
    summary_frame['Output_Dir'] = str(result['output_dir'])
    summary_frames.append(summary_frame)

comparison_frame = pd.concat(summary_frames, ignore_index=True)
comparison_frame = comparison_frame.sort_values(['Test_R2', 'Prediction_R2'], ascending=[False, False]).reset_index(drop=True)
comparison_frame
        


In [ ]:
display(comparison_frame)

best_test_row = comparison_frame.sort_values('Test_R2', ascending=False).iloc[0]
best_prediction_row = comparison_frame.sort_values('Prediction_R2', ascending=False).iloc[0]

print('Best held-out test architecture:')
print(best_test_row[['Architecture', 'Test_R2', 'Test_RMSE', 'Prediction_R2', 'Output_Dir']])
print()
print('Best prediction-set architecture:')
print(best_prediction_row[['Architecture', 'Prediction_R2', 'Test_R2', 'Test_RMSE', 'Output_Dir']])
        


In [ ]:
comparison_csv_path = output_dir / 'architecture_comparison_summary.csv'
comparison_frame.to_csv(comparison_csv_path, index=False)

figure, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_metrics = [
    ('Test_R2', 'Test R2', False),
    ('Test_RMSE', 'Test RMSE', True),
    ('Prediction_R2', 'Prediction R2', False),
]
bar_colors = ['#355070', '#6d597a', '#b56576', '#e56b6f']

for axis, (metric_key, title, ascending), color in zip(axes, plot_metrics, bar_colors[:3]):
    plot_frame = comparison_frame.sort_values(metric_key, ascending=ascending)
    axis.bar(plot_frame['Architecture'], plot_frame[metric_key], color=color, alpha=0.9)
    axis.set_title(title)
    axis.tick_params(axis='x', rotation=20)
    axis.grid(axis='y', alpha=0.3)

figure.tight_layout()
comparison_plot_path = output_dir / 'architecture_metric_comparison.png'
figure.savefig(comparison_plot_path, dpi=200, bbox_inches='tight')
plt.close(figure)

comparison_json_path = output_dir / 'best_architecture_summary.json'
best_payload = {
    'best_test_architecture': best_test_row.to_dict(),
    'best_prediction_architecture': best_prediction_row.to_dict(),
}
comparison_json_path.write_text(json.dumps(best_payload, indent=2), encoding='utf-8')

saved_files = sorted(path.name for path in output_dir.iterdir() if path.is_file())
print(f'Saved comparison summary to {comparison_csv_path}')
print(f'Saved comparison plot to {comparison_plot_path}')
print('Top-level saved files:')
for name in saved_files:
    print(f' - {name}')
print(f'Per-model outputs are under {per_model_output_root}')

if PUSH_RESULTS_TO_GITHUB:
    if not IN_COLAB:
        raise RuntimeError('GitHub auto-push is only intended for the Colab clone workflow.')
    if not GITHUB_TOKEN:
        raise ValueError('Set GITHUB_TOKEN before enabling PUSH_RESULTS_TO_GITHUB.')

    git_run_dir = git_output_root / output_dir.name
    if git_run_dir.exists():
        shutil.rmtree(git_run_dir)
    git_run_dir.mkdir(parents=True, exist_ok=True)

    top_level_files = [
        'architecture_comparison_summary.csv',
        'architecture_metric_comparison.png',
        'best_architecture_summary.json',
    ]
    for file_name in top_level_files:
        source_path = output_dir / file_name
        if source_path.is_file():
            shutil.copy2(source_path, git_run_dir / file_name)

    source_per_model_root = output_dir / 'per_model'
    destination_per_model_root = git_run_dir / 'per_model'
    if source_per_model_root.is_dir():
        for source_path in source_per_model_root.rglob('*'):
            relative_path = source_path.relative_to(source_per_model_root)
            destination_path = destination_per_model_root / relative_path
            if source_path.is_dir():
                destination_path.mkdir(parents=True, exist_ok=True)
                continue
            if not PUSH_MODEL_TO_GITHUB and source_path.name == 'lattice_gnn_model.pt':
                continue
            destination_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_path, destination_path)

    (git_output_root / 'latest_run.txt').write_text(str(git_run_dir.relative_to(repo_root)), encoding='utf-8')

    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.name', GIT_COMMIT_USERNAME], check=True)
    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.email', GIT_COMMIT_EMAIL], check=True)

    remote_url = subprocess.run(
        ['git', '-C', str(repo_root), 'remote', 'get-url', 'origin'],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    auth_url = remote_url.replace('https://', f'https://{GITHUB_TOKEN}@', 1)
    subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', auth_url], check=True)

    try:
        subprocess.run(['git', '-C', str(repo_root), 'add', str(git_run_dir), str(git_output_root / 'latest_run.txt')], check=True)
        diff_result = subprocess.run(
            ['git', '-C', str(repo_root), 'diff', '--cached', '--quiet'],
            check=False,
        )
        if diff_result.returncode == 0:
            print('No GitHub changes to commit.')
        else:
            commit_message = f'Add architecture comparison cloud results for {output_dir.name}'
            subprocess.run(['git', '-C', str(repo_root), 'commit', '-m', commit_message], check=True)
            subprocess.run(['git', '-C', str(repo_root), 'push', 'origin', GIT_BRANCH], check=True)
            print(f'Pushed results to GitHub under {git_run_dir.relative_to(repo_root)}')
    finally:
        subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', remote_url], check=True)
        


In [ ]:
if IN_COLAB and not SAVE_OUTPUTS_TO_DRIVE:
    from google.colab import files
    archive_path = '/content/gnn_outputs_architecture_comparison.zip'
    !cd /content && zip -qr gnn_outputs_architecture_comparison.zip gnn_outputs_architecture_comparison
    files.download(archive_path)
else:
    print(f'Outputs are in {output_dir}')
        
